In [0]:
import pandas as pd

In [0]:
temporal_path = "/Volumes/workspace/default/ucf_crime/annotations/temporal/temporal_clean.csv"
spatial_path = "/Volumes/workspace/default/ucf_crime/annotations/bounding-boxes/annotations_flat.csv"

temporal_df = pd.read_csv(temporal_path)
spatial_df = pd.read_csv(spatial_path)

display(temporal_df.head())
display(spatial_df.head())


In [0]:
temporal_df["video_id"] = temporal_df["video"].str.replace(".mp4", "", regex=False)
spatial_df["video_id"] = spatial_df["video_name"]

In [0]:
temporal_videos = set(temporal_df["video_id"])
spatial_videos = set(spatial_df["video_id"])

common_videos = temporal_videos & spatial_videos

print("Temporal videos:", len(temporal_videos))
print("Spatial videos:", len(spatial_videos))
print("Common videos:", len(common_videos))

In [0]:
common_df = pd.DataFrame(sorted(common_videos),columns=["video_id"])

temporal_common = temporal_df[temporal_df["video_id"].isin(common_videos)].copy()
spatial_common = spatial_df[spatial_df["video_id"].isin(common_videos)].copy()

In [0]:
# number of temporal intervals per class
interval_counts = (
    temporal_common["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="interval_count")
    )
display(interval_counts)

In [0]:
# number of unique videos per class
video_counts = (
    temporal_common
    .groupby("class")["video_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="video_count")
    )

display(video_counts)

In [0]:
# do annotation dfs agree on category?
class_check = (temporal_common[["video_id", "class"]].drop_duplicates().
               merge(
                   spatial_common[["video_id", "category"]].drop_duplicates(),
                   on="video_id",
                   how="inner"
                   )
               )

class_check["class_match"] = class_check["class"] == class_check["category"]

print("Total videos checked:", len(class_check))
print("Class mismatches:", (~class_check["class_match"]).sum())

mismatches = class_check[class_check["class_match"] == False]
if not mismatches.empty:
    display(mismatches)

In [0]:
# keep only videos that exist in both annotation sources
temporal_common = temporal_df[temporal_df["video_id"].isin(common_videos)].copy()
spatial_common = spatial_df[spatial_df["video_id"].isin(common_videos)].copy()

In [0]:
candidate_matches = spatial_common.merge(
    temporal_common[["video_id", "class", "interval_id", "start_frame", "end_frame"]], on="video_id", how="inner")

In [0]:
ground_truth_df = candidate_matches[
    (candidate_matches["frame_number"] >= candidate_matches["start_frame"]) &
    (candidate_matches["frame_number"] <= candidate_matches["end_frame"])
    ].copy()

ground_truth_df = ground_truth_df[
    [
        "split",
        "video_id",
        "class",
        "interval_id",
        "start_frame",
        "end_frame",
        "frame_number",
        "x1",
        "y1",
        "x2",
        "y2"
    ]
]

display(ground_truth_df.head(20))

In [0]:
print("Matched rows:", len(ground_truth_df))
print("Matched videos:", ground_truth_df["video_id"].nunique())

invalid_matches = ground_truth_df[
    (ground_truth_df["frame_number"] < ground_truth_df["start_frame"]) |
    (ground_truth_df["frame_number"] > ground_truth_df["end_frame"])
]

print("Invalid matches:", len(invalid_matches))

In [0]:
import os

output_dir = "/Volumes/workspace/default/ucf_crime/processed"
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(
    output_dir,
    "spatiotemporal_ground_truth.csv"
    )
ground_truth_df.to_csv(output_csv_path, index=False)
